# 4.4 — Programmatic waypoint tasks with TERRA

A waypoint task is **a scene, an actor and an ordered list of XY targets**. This notebook

1. runs the registered task `myoFullBodyWaypoint-v0` (MyoFullBody, random routes; CPU env + mjlab twin);
2. builds an obstacle course programmatically (stairs, beams, gaps) as an `MjSpec` edit;
3. drives the [TERRA-4B](https://huggingface.co/merc-s/TERRA-4B) policy ([TERRA](https://github.com/amathislab/terra), terrain-aware full-body motion tracking) through the course, natively in MyoSuite.

**Install:** `pip install -e ".[musclemimic]" orbax-checkpoint jupyter` (Orbax, and with it JAX, is only used to read the checkpoint).
The notebook downloads TERRA-4B at a pinned revision and one MuscleMimic walking clip from `amathislab/musclemimic-retargeted`, a gated dataset derived from AMASS: accept its terms on Hugging Face and log in (`hf auth login`); cite AMASS and KIT (see `myosuite/integrations/musclemimic/README.md`). Nothing is redistributed here.

In [ ]:
import os
import sys
from functools import partial
from itertools import product
from pathlib import Path

if sys.platform.startswith("linux") and not os.environ.get("DISPLAY"):
    os.environ.setdefault("MUJOCO_GL", "egl")

import imageio
import matplotlib.pyplot as plt
import mujoco
import numpy as np
from IPython.display import Image, Video, display

from myosuite import make_env
from myosuite.envs.waypoint import WaypointEnv, WaypointTaskCfg
from myosuite.terms.waypoint import WaypointRewardCfg

out_dir = Path("waypoint_outputs")
out_dir.mkdir(exist_ok=True)

## 1. The registered task

`myoFullBodyWaypoint-v0` draws a random 4-waypoint route at every reset (segments of 1–2 m, turns up to ±46°) in front of the MyoFullBody. The **reward** pays the distance closed toward the active waypoint, +1 per waypoint reached and −10 on a fall, so the return is bounded by the route length and ending an episode early never pays. The **observation** is `qpos` without the root's world `x, y`, `qvel · ctrl_dt`, muscle activations and the next two waypoints in the pelvis heading frame. Actions are raw muscle controls.

The goal logic is a set of shared term functions (`myosuite.terms.waypoint`) used by the CPU env and by its mjlab twin, so a policy trained on the GPU replays here.

In [ ]:
env = make_env("myoFullBodyWaypoint-v0")
obs, _ = env.reset(seed=0)
base = env.unwrapped
print("observation", obs.shape, {k: v.shape for k, v in base.get_obs_dict().items()})
print("route (m):", base.waypoints.round(2).tolist())
for _ in range(100):  # zero muscle drive: the body collapses
    obs, reward, terminated, truncated, info = env.step(np.zeros(env.action_space.shape))
    if terminated:
        break
print(f"after {base.data.time:.2f} s: failed={info['failed']}, reward={reward:.2f}")
env.close()

Train the same task on the GPU (mjlab twin, same `env_id`):

```bash
python scripts/train_mjlab.py myoFullBodyWaypoint-v0 --env.scene.num-envs 4096
```

## 2. Build a course programmatically

The scene is a function editing an `MjSpec`: ordinary static box geoms (`size` are half extents). They go in geom group 2 with the floor, where TERRA's heightmap looks for terrain, and use TERRA's terrain friction. Change the dimensions or write any other geometry (rotated boxes, meshes, heightfields) the same way.

In [ ]:
def course_scene(spec: mujoco.MjSpec, rise: float = 0.06, beam_width: float = 0.36, gap: float = 0.20) -> None:
    """Stairs up, a beam with a right-angle turn, a jump, stairs down and a second jump."""

    def box(name: str, x: float, y: float, width: float, length: float, height: float) -> None:
        spec.worldbody.add_geom(
            name=name, type=mujoco.mjtGeom.mjGEOM_BOX, group=2,
            pos=[x, y, height / 2], size=[width / 2, length / 2, height / 2],
            contype=1, conaffinity=1, condim=3, friction=[1.0, 0.1, 0.1],
            rgba=[0.27, 0.51, 0.71, 1],
        )

    for i in range(4):
        box(f"course_up_{i}", 0, -0.8 - (i + 0.5) * 0.3, 1, 0.3, (i + 1) * rise)
    top = 4 * rise
    box("course_top", 0, -2.3, 0.8, 0.6, top)
    box("course_beam_east", 1.1, -2.3, 1.4, beam_width, top)
    box("course_turn", 2.1, -2.3, 0.6, 0.8, top)
    box("course_beam_south", 2.1, -3.0, beam_width, 0.6, top)
    box("course_jump_landing", 2.1, -3.6 - gap, 0.8, 0.6, top)
    for i in range(3):
        box(f"course_down_{i}", 2.1, -4.05 - gap - i * 0.3, 1, 0.3, (3 - i) * rise)
    box("course_jump2_landing", 2.1, -5.1 - 2 * gap, 0.8, 0.6, rise)


waypoints = np.array([
    [0, -1.1], [0, -2.3], [1.05, -2.3], [2.1, -2.3],
    [2.1, -3.1], [2.1, -3.8], [2.1, -4.9], [2.1, -5.5], [2.1, -6.4],
])

Use the same environment ID for any course; constructor kwargs override its defaults. No additional registration is needed.

```python
from myosuite.core.config import EnvConfig

course = EnvConfig(
    "myoFullBodyWaypoint-v0", max_episode_steps=6000,
    task_kwargs={"edit_fn": course_scene,
                 "task": WaypointTaskCfg(waypoints=tuple(map(tuple, waypoints)), min_site_height=0.5)},
)
env_cpu = make_env(course)
env_gpu = make_env(course, backend="mjlab", num_envs=4096)
```

To load an XML scene instead, supply `model_path` (it replaces the default recipe); use `model_recipe` for another actor recipe.

## 3. The TERRA actor on the course

TERRA-4B was trained on `musclemimic_models` 1.0.5 with two muscle-wrap corrections and the MJX solver settings; `build_terrain_fullbody_spec` rebuilds exactly that actor and applies the scene. Supply this compiled model to `make_env("myoFullBodyWaypoint-v0", model=model)` for CPU playback. For the GPU twin, use a recipe or XML scene through the same environment ID.

In [ ]:
from myosuite.integrations.musclemimic import (
    MIMIC_SITES, TerrainController, load_terrain_policy, TerrainHeights,
    compose_waypoint_reference, download_terrain_checkpoint, build_terrain_fullbody_spec,
)

model = build_terrain_fullbody_spec(course_scene).compile()
data = mujoco.MjData(model)
mujoco.mj_forward(model, data)
heights = TerrainHeights(model)
print(f"TERRA actor: {model.nu} muscles, {model.nq} qpos; ground under the first step: {heights(data, [[0, -0.95]])[0]:.2f} m")

## 4. A reference walk along the route

TERRA tracks a full-body reference motion; it does not plan. `compose_waypoint_reference` is a small planner: it loops one gait cycle of a straight walking clip, steers it along the route (corners rounded over 30 cm) and lifts the root by the terrain height (gaps up to 60 cm bridged). Feet are not re-targeted onto steps: TERRA's heightmap has to handle the terrain.

In [ ]:
from huggingface_hub import hf_hub_download
from myosuite.core.trajectory_io import load_motion_clip

clip_path = hf_hub_download(
    "amathislab/musclemimic-retargeted", "MyoFullBody/gmr/KIT/167/walking_medium06_poses.npz", repo_type="dataset"
)
clip = load_motion_clip(Path(clip_path), expected_nq=model.nq, expected_nv=model.nv)
ctrl_dt = 0.01  # TERRA runs at 100 Hz: 0.002 s physics, 5 substeps
start = np.zeros(2)  # the route starts at the origin
reference = compose_waypoint_reference(
    model, clip, start, waypoints, ctrl_dt, lambda xy: heights(data, xy), MIMIC_SITES
)
print(f"reference: {reference.num_frames * ctrl_dt:.1f} s")

camera = mujoco.MjvCamera()
camera.lookat[:] = [1, -3.2, 0.3]
camera.distance, camera.azimuth, camera.elevation = 9, 135, -40
data.qpos[:] = reference.qpos[0]
mujoco.mj_forward(model, data)
with mujoco.Renderer(model, 480, 640) as renderer:
    renderer.update_scene(data, camera=camera)
    overview = renderer.render()
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(overview)
axes[0].axis("off")
axes[0].set_title("Course and TERRA actor at the reference start")
axes[1].plot(reference.qpos[:, 0], reference.qpos[:, 1], color="gray", label="reference root")
for i, point in enumerate(waypoints):
    axes[1].add_patch(plt.Circle(point, 0.3, color="orange", alpha=0.35))
    axes[1].annotate(str(i + 1), point)
axes[1].set(aspect="equal", xlabel="x (m)", ylabel="y (m)", title="Waypoints (30 cm radius)")
axes[1].legend()
plt.show()

## 5. Run TERRA in the waypoint env

The episode starts at the reference's first state, as TERRA's own evaluation. `TerrainController` builds TERRA's observation (heightmap, touch, reference goal) from the env's `MjData` and the reference frame, and returns the policy's muscle controls as the env action. The existing MuscleMimic runner loads the checkpoint, normalizes observations and samples actions. The adapter adds the checkpoint’s terrain and reference goal; the tracking goal is checked against pinned upstream fixtures. The env scores the route with its own criteria (arrival radius 30 cm, fall when the pelvis drops below 50 cm). Actions are sampled, as in TERRA's evaluation.

In [ ]:
policy = load_terrain_policy(download_terrain_checkpoint(), model, reference, stochastic=True, seed=0)

task = WaypointTaskCfg(
    site_name="pelvis_mimic", waypoints=tuple(map(tuple, waypoints)),
    arrival_radius=0.3, min_site_height=0.5, reward=WaypointRewardCfg(),
)
env = make_env(
    "myoFullBodyWaypoint-v0", model=model, task=task, initial_qpos=reference.qpos[0], initial_qvel=reference.qvel[0],
    frame_skip=5, render_mode="rgb_array",
)
env.reset(seed=0)
controller = TerrainController(policy, model, reference, env.unwrapped.frame_skip)
frames, ret = [], 0.0
with mujoco.Renderer(model, 360, 640) as renderer:
    for step in range(reference.num_frames - 1):
        _, reward, terminated, _, info = env.step(controller(env.unwrapped.data))
        ret += reward
        if step % 4 == 0:
            follow = mujoco.MjvCamera()
            follow.lookat[:] = env.unwrapped.data.qpos[:3]
            follow.distance, follow.azimuth, follow.elevation = 3.5, 135, -15
            renderer.update_scene(env.unwrapped.data, camera=follow)
            frames.append(renderer.render())
        if terminated:
            break
print(
    f"{env.unwrapped.data.time:.1f} s: {info['next_waypoint']}/{len(waypoints)} waypoints, "
    f"solved={info['solved']}, failed={info['failed']}, return={ret:.2f}"
)
video = out_dir / "terra_course.mp4"
imageio.mimsave(video, frames, fps=25)
preview = out_dir / "terra_course_preview.jpg"
imageio.imwrite(preview, frames[len(frames) // 2], quality=80)
display(Image(filename=str(preview), width=640))
print(f"Video saved to {video}; download it from Colab’s Files panel.")
EMBED_VIDEO = False  # Set True for inline playback; embeds the MP4 in saved outputs.
if EMBED_VIDEO:
    display(Video(str(video), embed=True, width=640))

The result is one sampled rollout on one course, with a simple reference planner: it is not a success rate. Change the seed, the course or the planner settings (`corner_radius`, `bridge`) and compare. A failure shows where the composed reference (flat-ground feet) or the policy reaches its limits; TERRA was trained on retargeted motions paired with their own terrain.

## 6. Variants

Scene, actor and goal vary independently. Compiling checks MuJoCo structure, not feasibility: re-plan the reference for each one.

In [ ]:
for rise, radius in product((0.04, 0.06), (0.2, 0.3)):
    variant = build_terrain_fullbody_spec(partial(course_scene, rise=rise)).compile()
    trial = WaypointEnv(variant, task=WaypointTaskCfg(waypoints=tuple(map(tuple, waypoints)), arrival_radius=radius))
    observation, _ = trial.reset(seed=0)
    assert trial.observation_space.contains(observation)
    print(f"step rise {rise:.2f} m, arrival radius {radius:.2f} m: {observation.shape[0]} observations")

The default preview is a small JPEG. Set `EMBED_VIDEO = True` for inline playback, or download the MP4 from the Files panel. Before saving a lightweight notebook, use **Edit → Clear all outputs** in Colab.